In [11]:
!pip install xgboost imbalanced-learn

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score, confusion_matrix

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [26]:
df = pd.read_csv("train_transaction.csv")

In [27]:
print(df.shape)
df.head()

(115971, 394)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
df = pd.read_csv("train_transaction.csv", nrows=100000)
print(df.shape)

(100000, 394)


In [41]:
print(df["isFraud"].value_counts())

isFraud
0    97439
1     2561
Name: count, dtype: int64


In [47]:
X = df.drop("isFraud", axis=1)
y = df["isFraud"]

In [48]:
X = X.select_dtypes(include="number")

In [49]:
X = X.drop("TransactionID", axis=1)

In [50]:
missing = X.isnull().mean()

X = X.drop(columns=missing[missing > 0.5].index)

In [51]:
X = X.fillna(X.median())
X = X.fillna(0)
print(X.isnull().sum().sum())

0


In [53]:
X_train, X_test, y_train, y_test = train_test_split(
	X, y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

In [54]:
print(X_train.shape)
print(X_test.shape)

(80000, 195)
(20000, 195)


In [55]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [56]:
print(y_train_smote.value_counts())

isFraud
0    77951
1    77951
Name: count, dtype: int64


In [57]:
model = XGBClassifier(
	n_estimators=100,
	max_depth=6,
	learning_rate=0.1,
	random_state=42,
	eval_metric="logloss"
)

model.fit(X_train_smote, y_train_smote)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [58]:
y_prob = model.predict_proba(X_test)[:, 1]

In [59]:
y_pred = (y_prob >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

ROC-AUC: 0.8865525820376641
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     19488
           1       0.73      0.43      0.54       512

    accuracy                           0.98     20000
   macro avg       0.86      0.71      0.77     20000
weighted avg       0.98      0.98      0.98     20000



In [60]:
print(confusion_matrix(y_test, y_pred))

[[19407    81]
 [  291   221]]


In [61]:
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
	y_pred = (y_prob >= threshold).astype(int)
	print(threshold, f1_score(y_test, y_pred))

0.1 0.34731810939989377
0.2 0.4495488105004102
0.3 0.5120967741935484
0.4 0.5409090909090909
0.5 0.542997542997543


In [62]:
thresholds = np.arange(0.1, 0.9, 0.05)

best_threshold = 0
best_f1 = 0

for threshold in thresholds:
	y_pred = (y_prob >= threshold).astype(int)
	f1 = f1_score(y_test, y_pred)

	if f1 > best_f1:
		best_f1 = f1
		best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

Best Threshold: 0.5000000000000001
Best F1: 0.542997542997543


In [63]:
y_pred_final = (y_prob >= best_threshold).astype(int)

print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     19488
           1       0.73      0.43      0.54       512

    accuracy                           0.98     20000
   macro avg       0.86      0.71      0.77     20000
weighted avg       0.98      0.98      0.98     20000



In [64]:
importance = pd.Series(
	model.feature_importances_,
	index=X.columns
)

print(importance.sort_values(ascending=False).head(10))

V30      0.127791
V317     0.070610
card3    0.070371
V57      0.056423
V128     0.036769
C2       0.035291
V133     0.030589
V70      0.029022
V287     0.027166
V318     0.018716
dtype: float32
